# Mesclagem RF e GB com os hiperparâmetros do artefato

Combina por média ponderada as previsões fora-da-dobra gravadas por `treino_rf_gb_otimizado.ipynb`,
que usam os hiperparâmetros de `colabs_S4/artefatos/hiperparametros.json`.

O método é o de `mesclagem_rf_gb.ipynb`, `varredura_pesos.ipynb` e `validacao_peso_otimo.ipynb`, sem
alteração: mesclagem por média ponderada componente a componente, varredura do peso do rf de 0 a 1 em
passos de 0,05, e peso ótimo escolhido pelo menor MAE macro por sufixo e área. Nenhum modelo é
treinado aqui.

## 1. Imports e caminhos

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_treinamento"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os, json

import pandas as pd
import numpy as np

PASTA_DADOS = "colabs_S3"

BASES = [
    ".", "..", os.path.join("..", ".."), os.path.join("..", "..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
]

def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )

print("Diretorio:", os.path.abspath("."))

## 2. Previsões e conferências

As previsões são lidas de `previsoes_rf_gb_otimizado.csv`. Duas conferências antes de qualquer
cálculo: as sete áreas têm de aparecer nos dois sufixos, e nenhum ensaio do conjunto de teste
reservado pode estar presente. O teste é identificado pela partição, onde seus ensaios ficam com
dobra ausente, o mesmo critério de `validacao_peso_otimo.ipynb`.

In [ ]:
previsoes = pd.read_csv(resolve(PASTA_DADOS, "previsoes_rf_gb_otimizado.csv"))
limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv")).set_index("tc_name")

AREAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
         "main_vent_sys", "mainback", "oven_hmi"]
SUFIXOS = ["_perfil", "_13cat"]

# As 7 areas precisam aparecer nos dois sufixos.
presenca = previsoes.groupby("sufixo")["area"].unique().apply(sorted)
assert set(presenca.index) == set(SUFIXOS), f"sufixos inesperados: {list(presenca.index)}"
for sufixo in SUFIXOS:
    assert list(presenca[sufixo]) == sorted(AREAS), f"{sufixo}: areas {list(presenca[sufixo])}"

# O conjunto de teste reservado nao pode estar nas previsoes: os run_idx de teste
# ficam com dobra ausente na particao, entao toda previsao tem de ter dobra.
with open(resolve("colabs_S3/preparacao_testes", "particao_treino_teste.json"), encoding="utf-8") as f:
    particao = json.load(f)
DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
dobras_das_previsoes = previsoes["run_idx"].map(DOBRA)
assert dobras_das_previsoes.notna().all(), "previsao sem dobra: ensaio fora do treino"
assert not set(previsoes["run_idx"]) & set(particao["run_idx_teste"]), "ensaio de teste presente"

print(f"{previsoes.shape[0]} previsoes | {len(SUFIXOS)} sufixos x {len(AREAS)} areas")
print(f"{dobras_das_previsoes.nunique()} dobras | "
      f"{len(particao['run_idx_teste'])} ensaios de teste reservados, nenhum presente")
previsoes.groupby(["sufixo", "area"])["componente"].nunique().to_frame("componentes").T

## 3. Métricas

Célula canônica de `colabs_S3/preparacao_testes/metricas.ipynb` (seção 1), copiada sem alteração, como nos
notebooks de mesclagem anteriores e em `treino_rf_gb_otimizado.ipynb`. As médias macro publicadas
adiante vêm de `consolidar(grupo="area")`.

In [ ]:
import os
import json

import numpy as np
import pandas as pd


MARGEM_KPI = 0.10   # margem de erro acordada no TAPI


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)


print("Metricas definidas: avaliar, consolidar")

## 4. Mesclagem

`mesclar` é a função dos notebooks anteriores: o peso é o do rf e o complemento é o do gb. As três
verificações fixam os casos que dão sentido à escala do peso.

In [ ]:
def mesclar(rf, gb, peso=0.5):
    """Combinacao ponderada das previsoes; peso e o do rf e (1 - peso) o do gb."""
    if not 0 <= peso <= 1:
        raise ValueError(f"peso fora de [0, 1]: {peso}")
    rf = np.asarray(rf, dtype=float)
    gb = np.asarray(gb, dtype=float)
    if rf.shape != gb.shape:
        raise ValueError(f"dimensoes diferentes: {rf.shape} e {gb.shape}")
    return peso * rf + (1 - peso) * gb


rf, gb = previsoes["rf"].to_numpy(), previsoes["gb"].to_numpy()
verificacao = pd.Series({
    "peso 0,5 = media simples": np.allclose(mesclar(rf, gb, 0.5), (rf + gb) / 2),
    "peso 0 = gb": np.array_equal(mesclar(rf, gb, 0), gb),
    "peso 1 = rf": np.array_equal(mesclar(rf, gb, 1), rf),
})
assert verificacao.all()
verificacao

## 5. Varredura do peso

Mesma grade de `varredura_pesos.ipynb`: 21 pesos de 0 a 1 em passos de 0,05. Cada peso é avaliado
componente a componente com `avaliar`, passando o limite de segurança de `limites_ativos.csv`, como
no baseline.

In [ ]:
def limite_de(col):
    lim = limites["limite_num"].get(col)
    return float(lim) if lim is not None and pd.notna(lim) else None


PESOS = np.round(np.linspace(0, 1, 21), 2)

linhas = []
for (sufixo, area, col), p in previsoes.groupby(["sufixo", "area", "componente"], sort=False):
    limite = limite_de(col)
    for peso in PESOS:
        linhas.append({"sufixo": sufixo, "area": area, "componente": col, "peso": peso,
                       **avaliar(p["real"], mesclar(p["rf"], p["gb"], peso), limite=limite)})

metricas = pd.DataFrame(linhas)
print(f"{len(PESOS)} pesos de {PESOS[0]:.2f} a {PESOS[-1]:.2f}, passo "
      f"{PESOS[1] - PESOS[0]:.2f} | {metricas.shape[0]} linhas por componente")

As métricas por componente viram métricas por área com `consolidar(grupo="area")`. Sufixo e
peso entram pelo laço, porque `consolidar` agrupa por uma coluna só. Os dois sufixos são tratados
separadamente do início ao fim: o peso é escolhido por sufixo e área, nunca em conjunto.

In [ ]:
# consolidar agrupa por uma coluna so; sufixo e peso entram pelo laco
blocos = []
for (sufixo, peso), bloco in metricas.groupby(["sufixo", "peso"], sort=False):
    blocos.append(consolidar(bloco, grupo="area").assign(sufixo=sufixo, peso=peso))

por_area = pd.concat(blocos, ignore_index=True)
print("por_area:", por_area.shape)

### Conferência dos extremos

O peso 0 tem de reproduzir o gb e o peso 1 o rf, ambos já calculados em `treino_rf_gb_otimizado.ipynb`
e gravados em `comparacao_otimizado_baseline.csv`. É a mesma conferência que `varredura_pesos.ipynb`
faz contra o baseline, e é ela que garante que a varredura roda sobre as mesmas previsões.

In [ ]:
comparacao_otimizado = pd.read_csv(resolve(PASTA_DADOS, "comparacao_otimizado_baseline.csv"))
CHAVE = ["sufixo", "area"]
MACRO = ["MAE_macro", "RMSE_macro", "R2_macro"]


def extremo_igual(peso, modelo):
    varrido = por_area[por_area["peso"] == peso].set_index(CHAVE)[MACRO].sort_index()
    ref = (comparacao_otimizado[comparacao_otimizado["modelo"] == modelo]
           .set_index(CHAVE)[[f"{m.replace('_macro', '')}_macro_otimizado" for m in MACRO]]
           .sort_index())
    return bool(np.allclose(varrido, ref.loc[varrido.index], equal_nan=True))


verificacao = pd.Series({
    "peso 0 = gb da secao anterior": extremo_igual(0.0, "gb"),
    "peso 1 = rf da secao anterior": extremo_igual(1.0, "rf"),
})
assert verificacao.all(), "os extremos da varredura nao reproduzem os modelos isolados"
verificacao

## 6. Peso ótimo por sufixo e área

O peso ótimo é o de menor MAE macro na varredura, escolhido separadamente para cada sufixo e área.
Em empate fica o menor peso, que é o comportamento de `idxmin` sobre a grade em ordem crescente,
igual ao dos notebooks anteriores.

In [ ]:
# em empate de MAE fica o primeiro peso da varredura, o menor
otimo = (por_area.loc[por_area.groupby(CHAVE)["MAE_macro"].idxmin(),
                      CHAVE + ["peso", "MAE_macro", "RMSE_macro", "R2_macro"]]
         .rename(columns={"peso": "peso_otimo"})
         .sort_values(CHAVE)
         .reset_index(drop=True))

assert len(otimo) == 14, f"esperados 7 areas x 2 sufixos, ha {len(otimo)}"
print("peso otimo por sufixo e area (peso do rf; o do gb e o complemento):")
otimo.pivot(index="area", columns="sufixo", values="peso_otimo")

## 7. As três configurações na mesma validação

RF, GB e a mesclagem no peso ótimo, com MAE, RMSE e R² macro por sufixo e área. RF e GB saem da
própria varredura, nos pesos 1 e 0, de modo que as três colunas vêm do mesmo cálculo.

In [ ]:
individuais = (por_area[por_area["peso"].isin([0.0, 1.0])]
               .assign(modelo=lambda d: np.where(d["peso"] == 1.0, "rf", "gb"))
               .pivot(index=CHAVE, columns="modelo", values=MACRO))

tres = pd.DataFrame(index=otimo.set_index(CHAVE).index)
o = otimo.set_index(CHAVE)
for rotulo, coluna in (("MAE", "MAE_macro"), ("RMSE", "RMSE_macro"), ("R2", "R2_macro")):
    tres[(rotulo, "rf")] = individuais[(coluna, "rf")]
    tres[(rotulo, "gb")] = individuais[(coluna, "gb")]
    tres[(rotulo, "mescla")] = o[coluna]
tres.columns = pd.MultiIndex.from_tuples(tres.columns)
tres.insert(0, ("peso", "otimo"), o["peso_otimo"])

print("RF, GB e mesclagem no peso otimo, na mesma validacao (medias macro):")
tres.round(3)

## 8. Mesclagem contra o melhor modelo individual

Para cada sufixo e área, a mesclagem é comparada ao modelo individual de menor MAE. A diferença é o
MAE da mesclagem menos o do melhor individual, em °C, negativa quando a mesclagem erra menos. A
classificação usa um limiar de 0,5 °C: melhora quando o MAE cai 0,5 °C ou mais, piora quando sobe
0,5 °C ou mais, empate quando a diferença absoluta fica abaixo disso.

In [ ]:
LIMIAR = 0.5   # °C

classificacao = o[["peso_otimo"]].copy()
classificacao["MAE_rf"] = individuais[("MAE_macro", "rf")]
classificacao["MAE_gb"] = individuais[("MAE_macro", "gb")]
classificacao["MAE_mescla"] = o["MAE_macro"]
classificacao["melhor_individual"] = np.where(
    classificacao["MAE_rf"] <= classificacao["MAE_gb"], "rf", "gb")
classificacao["MAE_melhor_individual"] = classificacao[["MAE_rf", "MAE_gb"]].min(axis=1)
# negativo quando a mesclagem erra menos que o melhor modelo individual
classificacao["delta_MAE"] = (classificacao["MAE_mescla"]
                              - classificacao["MAE_melhor_individual"])
classificacao["classificacao"] = np.select(
    [classificacao["delta_MAE"] <= -LIMIAR, classificacao["delta_MAE"] >= LIMIAR],
    ["melhora", "piora"], default="empate")
classificacao = classificacao.reset_index().sort_values(CHAVE).reset_index(drop=True)

# As 7 areas precisam aparecer nos dois sufixos.
presenca = classificacao.groupby("sufixo")["area"].nunique()
assert set(presenca.index) == set(SUFIXOS), f"sufixos inesperados: {list(presenca.index)}"
assert (presenca == 7).all(), f"area faltando em algum sufixo:\n{presenca}"
assert len(classificacao) == 14, f"esperadas 14 linhas, ha {len(classificacao)}"

print("assert OK: 7 areas nos dois sufixos |", len(classificacao), "linhas")
print("contagem:", classificacao["classificacao"].value_counts().to_dict())
classificacao.round(3)

## 9. Gravação

Mesma pasta dos demais CSVs de resultado.

In [ ]:
tabela = classificacao.copy()
for rotulo, coluna in (("RMSE", "RMSE_macro"), ("R2", "R2_macro")):
    tabela[f"{rotulo}_rf"] = individuais[(coluna, "rf")].reindex(
        pd.MultiIndex.from_frame(tabela[CHAVE])).to_numpy()
    tabela[f"{rotulo}_gb"] = individuais[(coluna, "gb")].reindex(
        pd.MultiIndex.from_frame(tabela[CHAVE])).to_numpy()
    tabela[f"{rotulo}_mescla"] = o[coluna].reindex(
        pd.MultiIndex.from_frame(tabela[CHAVE])).to_numpy()

tabela = tabela[["sufixo", "area", "peso_otimo",
                 "MAE_rf", "MAE_gb", "MAE_mescla",
                 "RMSE_rf", "RMSE_gb", "RMSE_mescla",
                 "R2_rf", "R2_gb", "R2_mescla",
                 "melhor_individual", "MAE_melhor_individual", "delta_MAE", "classificacao"]]

pasta = os.path.dirname(resolve(PASTA_DADOS, "limites_ativos.csv"))
caminho = os.path.join(pasta, "comparacao_mesclagem_otimizada.csv")
tabela.to_csv(caminho, index=False)

print(caminho)
print(f"{tabela.shape[0]} linhas, {tabela.shape[1]} colunas")
tabela.round(3)

## 10. Leitura dos resultados

A mesclagem não supera o melhor modelo individual em nenhuma área. As catorze combinações de área e sufixo ficaram como empate, porque nenhuma diferença chegou a 0,5 °C.

Em dez combinações o peso ótimo ficou num extremo da grade: 1,00 em CT Knobs, Main Vent Sys, Mainback e Oven HMI nos dois sufixos e em IPC Cooling no `_perfil`, e 0,00 em IPC Comp no `_13cat`. Nesses casos a mesclagem é o próprio Random Forest ou o próprio Gradient Boosting.

Nas outras quatro o peso é intermediário, mas o ganho é pequeno. Coil Temps teve a maior queda de MAE, 0,159 °C no `_13cat` (peso 0,70) e 0,148 °C no `_perfil` (peso 0,80). Em IPC Cooling no `_13cat` e em IPC Comp no `_perfil`, a queda ficou abaixo de 0,01 °C.

A ausência de pioras não favorece a mesclagem. Como a grade inclui os pesos 0 e 1, o peso escolhido nunca erra mais que o melhor modelo individual.

O peso foi escolhido e avaliado nas mesmas dezessete dobras, então os ganhos acima são otimistas. Para medi-los sem esse viés, o peso teria de ser escolhido fora da dobra em que é avaliado.